# Regional Obligations: Silver

Cleans `pondolens.dev_bronze.regional_expenditure_obligations` and merges it into `pondolens.dev_silver.regional_obligations`.

**Silver columns:** `obligation_id | year | category | particular | region_code | region_name | expenditure_amount | processing_run_id | processed_at`

**Rules**
- **Business key:** `year, category, particular, region_code`. The MERGE matches on this key.
- **Regions:** every source spelling (`Region IV-A`, `Region-IVA`, `REGION IVA`, …) maps to one `region_code` (`IVA`) and one `region_name` (`CALABARZON`) through the team's region reference in step 1 (same codes as GRDP Silver). An unknown spelling stops the run instead of creating a new region.
- **Non-regional allocations:** `NW` = Nationwide (undistributed funds slated for later allocation); `CO` = Central Office (managed directly by the agency's head office). Exclude them in Gold when comparing regions.
- **`Total` rows are removed.** Total = Nationwide + Central Office + all regions, so it can always be recomputed with `SUM`. Keeping it would double-count.
- `obligation_id`: existing records keep their ID; only new records get `MAX(obligation_id) + ROW_NUMBER()` (2,000,000 when empty).
- `expenditure_amount`: `-` becomes 0, blanks stay NULL, thousand pesos × 1000.
- **Snapshot reconciliation:** for each year in Bronze, Silver records no longer in Bronze are deleted. Stops if more than 10% of in-scope records would be deleted.
- **Idempotent:** rerunning with unchanged Bronze writes 0 rows.
- **Safe to run at the same time:** a run lock (step 3) lets only one run write. A second run stops before writing with *"Another run is in progress"*; just run it again later.
- **Upgrades an older table automatically:** step 4 converts the old `region` column (or older codes) to the current codes and removes `Total` rows, keeping every other `obligation_id`.

## 1. Region reference

Team region codes and names (GRDP Silver list + `NW` and `CO` from Regional GAA), plus one optional extra spelling. Each value is matched after removing spaces, dashes and case, so `Region IV-A`, `Region-IVA` and `REGION IVA` all become the key `REGIONIVA`. Add a row (or an extra spelling) here if a new source spelling appears.

`Total` is not in the reference on purpose: it is filtered out in step 5.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW region_ref AS
SELECT
  region_code,
  region_name,
  EXPLODE(ARRAY_DISTINCT(FILTER(
    ARRAY(region_code, CONCAT('REGION', region_code), UPPER(REGEXP_REPLACE(region_name, '[^A-Za-z0-9]', '')), extra_alias),
    a -> a IS NOT NULL
  ))) AS alias_key
FROM VALUES
  ('NW',    'Nationwide',                                      'NATIONAWIDE'),
  ('CO',    'Central Office',                                  NULL),
  ('NCR',   'National Capital Region',                         NULL),
  ('CAR',   'Cordillera Administrative Region',                NULL),
  ('I',     'Ilocos Region',                                   NULL),
  ('II',    'Cagayan Valley',                                  NULL),
  ('III',   'Central Luzon',                                   NULL),
  ('IVA',   'CALABARZON',                                      NULL),
  ('IVB',   'MIMAROPA Region',                                 'MIMAROPA'),
  ('V',     'Bicol Region',                                    NULL),
  ('VI',    'Western Visayas',                                 NULL),
  ('NIR',   'Negros Island Region',                            NULL),
  ('VII',   'Central Visayas',                                 NULL),
  ('VIII',  'Eastern Visayas',                                 NULL),
  ('IX',    'Zamboanga Peninsula',                             NULL),
  ('X',     'Northern Mindanao',                               NULL),
  ('XI',    'Davao Region',                                    NULL),
  ('XII',   'SOCCSKSARGEN',                                    NULL),
  ('XIII',  'Caraga',                                          NULL),
  ('BARMM', 'Bangsamoro Autonomous Region in Muslim Mindanao', 'BANGSAMORO'),
  ('PHL',   'Philippines',                                     NULL)
AS r(region_code, region_name, extra_alias);

-- Every spelling must point to exactly one region.
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT alias_key), 'Region reference has a spelling mapped to two regions.')
FROM region_ref;

## 2. Create the Silver table and the run-lock table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.regional_obligations (
  obligation_id INT NOT NULL,
  year INT NOT NULL,
  category STRING NOT NULL,
  particular STRING,
  region_code STRING NOT NULL COMMENT 'NCR, CAR, I-XIII, IVA, IVB, NIR, BARMM, PHL; non-regional: NW (Nationwide), CO (Central Office)',
  region_name STRING NOT NULL,
  expenditure_amount DECIMAL(20,2) COMMENT 'Pesos',
  processing_run_id STRING NOT NULL,
  processed_at TIMESTAMP NOT NULL,
  CONSTRAINT regional_obligations_pk PRIMARY KEY (obligation_id)
)
USING DELTA
COMMENT 'Regional expenditure obligations (actual obligations) in pesos. Grain: year, category, particular, region_code.';

CREATE SCHEMA IF NOT EXISTS pondolens.dev_gold;
-- Deletion vectors + row tracking enable row-level concurrency, so locks for
-- different tables (rows) do not block each other.
CREATE TABLE IF NOT EXISTS pondolens.dev_gold.etl_run_lock (table_name STRING NOT NULL, locked_by STRING, locked_at TIMESTAMP)
USING DELTA
TBLPROPERTIES ('delta.enableDeletionVectors' = 'true', 'delta.enableRowTracking' = 'true')
COMMENT 'Run lock: one row per target table; prevents two runs writing the same table at once.';

MERGE INTO pondolens.dev_gold.etl_run_lock AS l
USING (SELECT 'regional_obligations' AS table_name) AS s
ON l.table_name = s.table_name
WHEN NOT MATCHED THEN INSERT (table_name, locked_by, locked_at) VALUES (s.table_name, NULL, NULL);

## 3. Run metadata and run lock

- Stops with **"Another run is in progress"** if someone else is running this notebook: wait, then run again.
- A **concurrent update** error means two runs claimed the lock at the same instant and the other won. Nothing was written; run again.
- A lock left by a failed run expires after 10 minutes (or use the manual unlock cell at the bottom).

In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_run_id STRING;
DECLARE OR REPLACE VARIABLE v_run_ts TIMESTAMP;
DECLARE OR REPLACE VARIABLE v_scope_years ARRAY<INT>;
SET VAR v_run_id = REPLACE(UUID(), '-', ''), v_run_ts = CURRENT_TIMESTAMP();

-- Both runs update the SAME row, so Delta lets only one commit.
UPDATE pondolens.dev_gold.etl_run_lock
SET locked_by = v_run_id, locked_at = CURRENT_TIMESTAMP()
WHERE table_name = 'regional_obligations'
  AND (locked_by IS NULL OR locked_at < CURRENT_TIMESTAMP() - INTERVAL 10 MINUTES);

SELECT ASSERT_TRUE(
  COUNT(*) > 0 AND COUNT_IF(locked_by = v_run_id) = COUNT(*),
  'Another run is in progress on regional_obligations. Wait for it to finish, then run again.'
)
FROM pondolens.dev_gold.etl_run_lock
WHERE table_name = 'regional_obligations';

SELECT v_run_id AS processing_run_id, v_run_ts AS processed_at;

## 4. Upgrade an older table (automatic, safe to rerun)

- **Old `region` column** (runs once): removes `Total` rows, adds `region_code` and `region_name`, fills them from the old values using the step 1 reference, then drops `region`. Dropping a column needs Delta column mapping, so it is enabled first. When not needed, each statement is a no-op (`SELECT 1`).
- **Every run:** removes any `Total` rows and corrects codes or names that differ from the reference (for example an older `NATIONWIDE` code becomes `NW`). Writes nothing when the table is already correct.

**Every remaining `obligation_id` and amount is kept.**

In [0]:
%sql
DECLARE OR REPLACE VARIABLE v_migrate BOOLEAN;
DECLARE OR REPLACE VARIABLE v_sql STRING;

SET VAR v_migrate = (
  SELECT COUNT_IF(column_name = 'region') = 1 AND COUNT_IF(column_name = 'region_code') = 0
  FROM pondolens.information_schema.columns
  WHERE table_schema = 'dev_silver' AND table_name = 'regional_obligations'
);
SELECT v_migrate AS upgrading_old_table;

SET VAR v_sql = IF(v_migrate, "ALTER TABLE pondolens.dev_silver.regional_obligations SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5')", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

SET VAR v_sql = IF(v_migrate, "DELETE FROM pondolens.dev_silver.regional_obligations WHERE UPPER(REGEXP_REPLACE(region, '[^A-Za-z0-9]', '')) = 'TOTAL'", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

SET VAR v_sql = IF(v_migrate, "ALTER TABLE pondolens.dev_silver.regional_obligations ADD COLUMNS (region_code STRING AFTER particular, region_name STRING AFTER region_code)", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

SET VAR v_sql = IF(v_migrate, "MERGE INTO pondolens.dev_silver.regional_obligations AS t USING region_ref AS r ON r.alias_key = UPPER(REGEXP_REPLACE(t.region, '[^A-Za-z0-9]', '')) WHEN MATCHED THEN UPDATE SET t.region_code = r.region_code, t.region_name = r.region_name", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

-- Every old row must have been mapped before the old column is dropped.
SELECT ASSERT_TRUE(COUNT_IF(region_code IS NULL OR region_name IS NULL) = 0,
  'Some old region values could not be mapped. Add them to the region reference (step 1) and rerun.')
FROM pondolens.dev_silver.regional_obligations;

SET VAR v_sql = IF(v_migrate, "ALTER TABLE pondolens.dev_silver.regional_obligations ALTER COLUMN region_code SET NOT NULL", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

SET VAR v_sql = IF(v_migrate, "ALTER TABLE pondolens.dev_silver.regional_obligations ALTER COLUMN region_name SET NOT NULL", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

SET VAR v_sql = IF(v_migrate, "ALTER TABLE pondolens.dev_silver.regional_obligations DROP COLUMN region", "SELECT 1");
EXECUTE IMMEDIATE v_sql;

-- Every run: remove Total rows and align codes/names with the reference (no-op when already correct).
DELETE FROM pondolens.dev_silver.regional_obligations WHERE region_code = 'TOTAL';

MERGE INTO pondolens.dev_silver.regional_obligations AS t
USING region_ref AS r
ON r.alias_key = UPPER(REGEXP_REPLACE(t.region_code, '[^A-Za-z0-9]', ''))
WHEN MATCHED AND (t.region_code <> r.region_code OR t.region_name <> r.region_name) THEN UPDATE SET
  t.region_code = r.region_code,
  t.region_name = r.region_name,
  t.processing_run_id = v_run_id,
  t.processed_at = v_run_ts;

## 5. Clean Bronze into a staging view

Trims text, removes `Total` rows, maps regions through the reference, keeps only the latest ingestion of each record, and converts amounts to pesos.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW regional_obligation_stage AS
WITH cleaned AS (
  SELECT
    TRY_CAST(TRIM(CAST(b.year AS STRING)) AS INT) AS year,
    NULLIF(REGEXP_REPLACE(TRIM(b.category), r'\s+', ' '), '') AS category,
    NULLIF(REGEXP_REPLACE(TRIM(b.particulars), r'\s+', ' '), '') AS particular,
    NULLIF(TRIM(b.regions), '') AS region_raw,
    r.region_code,
    r.region_name,
    LOWER(TRIM(b.unit_amount)) AS unit_amount,
    NULLIF(REPLACE(TRIM(CAST(b.expenditure_amount AS STRING)), ',', ''), '') AS amount_text,
    b._ingested_at
  FROM pondolens.dev_bronze.regional_expenditure_obligations AS b
  LEFT JOIN region_ref AS r
    ON r.alias_key = UPPER(REGEXP_REPLACE(b.regions, '[^A-Za-z0-9]', ''))
  -- Total = Nationwide + Central Office + all regions; recompute with SUM instead of storing it.
  WHERE NOT (UPPER(REGEXP_REPLACE(b.regions, '[^A-Za-z0-9]', '')) <=> 'TOTAL')
),
latest AS (
  -- Latest ingestion per business key. RANK keeps ties, so conflicting values
  -- inside the same ingestion are caught by the duplicate-key check in step 6.
  SELECT * FROM cleaned
  QUALIFY RANK() OVER (PARTITION BY year, category, particular, region_code ORDER BY _ingested_at DESC) = 1
)
SELECT DISTINCT
  year, category, particular, region_code, region_name, unit_amount, amount_text,
  IF(region_code IS NULL, COALESCE(region_raw, '(blank)'), NULL) AS unmapped_region,
  CASE
    WHEN amount_text IN ('-', '–', '—') THEN CAST(0 AS DECIMAL(20,2))
    WHEN unit_amount = 'thousand pesos' THEN CAST(TRY_CAST(amount_text AS DECIMAL(20,2)) * 1000 AS DECIMAL(20,2))
    WHEN unit_amount = 'pesos' THEN TRY_CAST(amount_text AS DECIMAL(20,2))
  END AS expenditure_amount
FROM latest;

## 6. Data quality checks

The first query lists any region spelling not in the reference (empty on a normal run). Each `ASSERT_TRUE` stops the run with its own message.

In [0]:
%sql
SELECT DISTINCT unmapped_region FROM regional_obligation_stage WHERE unmapped_region IS NOT NULL;

SELECT
  ASSERT_TRUE(COUNT(*) > 0, 'Bronze returned no rows.') AS has_rows,
  ASSERT_TRUE(COUNT(*) = COUNT(year) AND COUNT(*) = COUNT(category),
    'Missing year or category in Bronze.') AS keys_present,
  ASSERT_TRUE(COUNT(*) = COUNT(region_code),
    'Blank or unmapped region (see the list above). Add the spelling to the region reference in step 1.') AS regions_mapped,
  ASSERT_TRUE(COUNT_IF(year NOT BETWEEN 2000 AND 2100) = 0, 'Year outside 2000-2100.') AS valid_year,
  ASSERT_TRUE(COUNT_IF(unit_amount IS NULL OR unit_amount NOT IN ('thousand pesos', 'pesos')) = 0,
    'Unknown unit_amount; update the conversion rule.') AS known_unit,
  ASSERT_TRUE(COUNT_IF(amount_text IS NOT NULL AND expenditure_amount IS NULL) = 0,
    'Some expenditure_amount values could not be parsed as numbers.') AS amounts_parsed,
  ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT STRUCT(year, category, particular, region_code)),
    'Duplicate business key: two source rows map to the same year/category/particular/region_code.') AS unique_key
FROM regional_obligation_stage;

## 7. Check the existing Silver table

In [0]:
%sql
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT obligation_id)
  AND COUNT(*) = COUNT(DISTINCT STRUCT(year, category, particular, region_code))
  AND COUNT_IF(obligation_id <= 2000000) = 0,
  'Silver has duplicate IDs, duplicate keys or IDs at or below 2000000.'
)
FROM pondolens.dev_silver.regional_obligations;

## 8. Snapshot reconciliation scope

Refresh scope = the years in this Bronze snapshot. Lists Silver records in those years that are no longer in Bronze (they will be deleted), and stops if that is more than 10% of in-scope records.

In [0]:
%sql
SET VAR v_scope_years = (SELECT ARRAY_SORT(COLLECT_SET(year)) FROM regional_obligation_stage);

CREATE OR REPLACE TEMP VIEW regional_obligation_removed AS
SELECT t.*
FROM pondolens.dev_silver.regional_obligations AS t
LEFT ANTI JOIN regional_obligation_stage AS s
  ON t.year = s.year
  AND t.category = s.category
  AND t.particular <=> s.particular
  AND t.region_code = s.region_code
WHERE ARRAY_CONTAINS(v_scope_years, t.year);

SELECT
  v_scope_years AS refresh_scope_years,
  r.rows_to_delete,
  c.silver_rows_in_scope,
  ASSERT_TRUE(r.rows_to_delete <= c.silver_rows_in_scope * 0.10,
    'More than 10% of in-scope Silver records would be deleted. Check that Bronze is complete.') AS delete_guard
FROM (SELECT COUNT(*) AS rows_to_delete FROM regional_obligation_removed) AS r
CROSS JOIN (SELECT COUNT(*) AS silver_rows_in_scope FROM pondolens.dev_silver.regional_obligations WHERE ARRAY_CONTAINS(v_scope_years, year)) AS c;

SELECT * FROM regional_obligation_removed ORDER BY obligation_id;

## 9. Assign obligation_id to new records only

`ROW_NUMBER()` runs only on business keys not yet in Silver and continues from the last allocated ID. Existing IDs never change.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW regional_obligation_merge_source AS
WITH new_ids AS (
  SELECT
    s.year, s.category, s.particular, s.region_code,
    l.last_id + ROW_NUMBER() OVER (ORDER BY s.year, s.category, s.particular, s.region_code) AS obligation_id
  FROM regional_obligation_stage AS s
  LEFT ANTI JOIN pondolens.dev_silver.regional_obligations AS t
    ON s.year = t.year
  AND s.category = t.category
  AND s.particular <=> t.particular
  AND s.region_code = t.region_code
  CROSS JOIN (SELECT COALESCE(MAX(CAST(obligation_id AS BIGINT)), 2000000) AS last_id FROM pondolens.dev_silver.regional_obligations) AS l
)
-- Existing records get NULL here; the MERGE never uses it for them.
SELECT s.year, s.category, s.particular, s.region_code, s.region_name, s.expenditure_amount, n.obligation_id
FROM regional_obligation_stage AS s
LEFT JOIN new_ids AS n
  ON s.year = n.year
  AND s.category = n.category
  AND s.particular <=> n.particular
  AND s.region_code = n.region_code;

SELECT ASSERT_TRUE(
  COUNT_IF(obligation_id IS NOT NULL AND obligation_id NOT BETWEEN 2000001 AND 2147483647) = 0,
  'New obligation_id values must be between 2000001 and 2147483647.'
)
FROM regional_obligation_merge_source;

## 10. Merge into Silver using the business key

- **Matched:** update only if the amount or region name changed. The ID stays the same.
- **Not matched:** insert with the new ID from step 9.
- **Not matched by source** (within the step 8 scope): delete.

The first statement re-checks the lock right before writing.

In [0]:
%sql
SELECT ASSERT_TRUE(
  COUNT(*) > 0 AND COUNT_IF(locked_by = v_run_id) = COUNT(*),
  'This run no longer holds the lock (it expired or was released). Nothing was written; run the notebook again.'
)
FROM pondolens.dev_gold.etl_run_lock
WHERE table_name = 'regional_obligations';

MERGE INTO pondolens.dev_silver.regional_obligations AS t
USING regional_obligation_merge_source AS s
ON t.year = s.year
  AND t.category = s.category
  AND t.particular <=> s.particular
  AND t.region_code = s.region_code
WHEN MATCHED AND (
  NOT (t.expenditure_amount <=> s.expenditure_amount)
  OR NOT (t.region_name <=> s.region_name)
) THEN UPDATE SET
  t.expenditure_amount = s.expenditure_amount,
  t.region_name = s.region_name,
  t.processing_run_id = v_run_id,
  t.processed_at = v_run_ts
WHEN NOT MATCHED THEN INSERT (
  obligation_id, year, category, particular, region_code, region_name,
  expenditure_amount, processing_run_id, processed_at
) VALUES (
  CAST(s.obligation_id AS INT), s.year, s.category, s.particular, s.region_code, s.region_name,
  s.expenditure_amount, v_run_id, v_run_ts
)
WHEN NOT MATCHED BY SOURCE AND ARRAY_CONTAINS(v_scope_years, t.year) THEN DELETE;

## 11. Post-merge checks and run summary

In [0]:
%sql
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT obligation_id) AND COUNT(*) = COUNT(DISTINCT STRUCT(year, category, particular, region_code)),
  'Duplicate IDs or business keys after the merge.'
)
FROM pondolens.dev_silver.regional_obligations;

-- Every staged record must now be in Silver with the same amount.
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Silver does not match the staged Bronze records.')
FROM regional_obligation_stage AS s
LEFT ANTI JOIN pondolens.dev_silver.regional_obligations AS t
  ON s.year = t.year
  AND s.category = t.category
  AND s.particular <=> t.particular
  AND s.region_code = t.region_code
  AND s.expenditure_amount <=> t.expenditure_amount;

SELECT
  year,
  COUNT(*) AS records,
  COUNT_IF(processing_run_id = v_run_id) AS written_this_run,
  COUNT_IF(expenditure_amount IS NULL) AS null_amounts,
  MIN(obligation_id) AS min_id,
  MAX(obligation_id) AS max_id
FROM pondolens.dev_silver.regional_obligations
GROUP BY year
ORDER BY year;

SELECT region_code, region_name, COUNT(*) AS records
FROM pondolens.dev_silver.regional_obligations
GROUP BY region_code, region_name
ORDER BY region_code;

## 12. Release the run lock

In [0]:
%sql
UPDATE pondolens.dev_gold.etl_run_lock
SET locked_by = NULL, locked_at = NULL
WHERE table_name = 'regional_obligations' AND locked_by = v_run_id;

## Manual unlock (only if a run failed halfway)

A failed run's lock expires after 10 minutes. To rerun sooner, **first make sure nobody else is running this notebook**, then uncomment and run this cell.

In [0]:
%sql
-- UPDATE pondolens.dev_gold.etl_run_lock
-- SET locked_by = NULL, locked_at = NULL
-- WHERE table_name = 'regional_obligations';